# 使用 Docker 部署的 Ollama

## 什么是 Ollama？

Ollama 是一个轻量级的本地大模型运行框架，支持在本地运行 Llama、Qwen、Gemma 等开源模型。

## Docker 部署 Ollama

```bash
# 拉取 Ollama 镜像
docker pull ollama/ollama

# 启动容器（映射 11434 端口）
docker run -d --name ollama -p 11434:11434 ollama/ollama

# 进入容器下载模型
docker exec -it ollama ollama pull qwen2.5:0.5b
```

> 也可以使用 GPU 版本：`docker run -d --gpus=all -p 11434:11434 ollama/ollama`

## 1. 基础连接与调用

使用 `langchain-ollama` 连接本地 Ollama 服务。

In [2]:
from langchain_ollama import ChatOllama

# 连接本地 Ollama（默认 http://localhost:11434）
llm = ChatOllama(
    model="qwen2.5:0.5b",
    base_url="http://localhost:11434",
)

# 非流式调用
response = llm.invoke("你好，简单介绍一下你自己,并且背诵滕王阁序")
print(response.content)

我是来自阿里云的Qwen模型。我叫范晓亮。我有一个非常独特、又充满魅力的名字——范晓亮。我的个性和性格是多变的，有智慧、机智的一面，但也有情感脆弱的一面。我学习能力强，擅长于语言理解和生成。在过去的几个月里，我被用作Qwen模型，负责回答各种问题，如文学、历史、科技等。

关于滕王阁序的内容，我将为您简单介绍：

《滕王阁序》是唐代诗人杜牧的代表作品之一，是山水诗中的一篇佳作。该文通过对滕王阁的描述和吟咏，描绘了一幅色彩鲜明、意境深远的画面。全篇共80字左右，用简洁而富有节奏感的语言表达出了作者的情感和思想。

如果您有任何问题或需要帮助，请随时告诉我，我会尽力提供支持和解答。希望您在使用过程中遇到的问题都能得到满意的答案！


## 2. 流式输出

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:0.5b")

print("流式输出：")
for chunk in llm.stream("用一句话解释什么是机器学习"):
    print(chunk.content, end="", flush=True)

print()

流式输出：
机器学习是通过算法让计算机从数据中学习并自动调整其行为，从而在不被显式编程的情况下做出决策和行动。这种方法依赖于大量的数据集和统计模型来改进性能和效率，使得计算机能够像人类一样分析、理解以及处理大量复杂的数据。


## 3. 多轮对话

使用 `SystemMessage` 和 `HumanMessage` 构建多轮对话。

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="qwen2.5:0.5b")

messages = [
    SystemMessage(content="你是一个Python编程助手，回答要简洁。"),
    HumanMessage(content="Python中如何读取JSON文件？"),
]

# response = llm.invoke(messages)
for chunk in llm.stream(messages):
    print(chunk.content,end="",flush=True)
# print(response.content)

在Python中，你可以使用内置的`json`模块来读取和写入JSON数据。以下是一些示例：

```python
import json

# 读取JSON文件
with open('file.json', 'r') as f:
    data = json.load(f)

# 写入JSON文件
f_out = open('output.json', 'w')
json.dump(data, f_out)
```

在上面的示例中，`'file.json'`是你要读取和写入的JSON文件的位置。`'output.json'`是输出到文件的位置。

此外，在处理数据时，请确保你的Python环境具有足够的权限来读取并写入文件。如果需要添加权限或修改现有权限，请参考Python的系统设置文档。

请注意，要读取的数据应该是用`json.dumps()`或者类似的函数创建的JSON字符串，而不是直接使用`open('file.json', 'r')`来打开一个文本文件，这样可以避免意外地覆盖原始数据或可能导致意外的结果。

## 4. 使用提示词模板

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOllama(model="qwen2.5:0.5b")

prompt = ChatPromptTemplate.from_template(
    "请用一句话简单介绍{topic}是什么？"
)

chain = prompt | llm
response = chain.invoke({"topic": "Docker容器技术"})
print(response.content)

## 5. 使用 Ollama Embedding 模型

Ollama 也支持本地运行 Embedding 模型，可用于 RAG 等场景。

In [ ]:
from langchain_ollama import OllamaEmbeddings

# 需要先下载 embedding 模型：docker exec -it ollama ollama pull nomic-embed-text
embeddings = OllamaEmbeddings(model="nomic-embed-text")

text = "LangChain是一个LLM应用开发框架"
vector = embeddings.embed_query(text)
print(f"向量维度: {len(vector)}")
print(f"前5个值: {vector[:5]}")